# Stitching together `.cha` files from the checkpoints

Perhaps inadvertently, I've saved the outputs for each conversation as a CEDA checkpoint. We need to iterate through them and stitch together the final file for analysis.

In [1]:
import os
import torch
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

In [2]:
DATA_PATH = '../data'

OUTPUT_PATH = os.path.join(DATA_PATH, 'lme_ready')
if not os.path.exists(OUTPUT_PATH):
    os.mkdir(OUTPUT_PATH)

OUTPUT_NAME = os.path.join(OUTPUT_PATH,'null-ceda-analysis.parquet')

In [3]:
def grab_all_files(PATH, file_type='.cha'):
    files = [
        [
            os.path.join(root, f) for f in files 
            if f.endswith(file_type) and (not f.startswith('._'))
        ] 
        for root, _, files in os.walk(PATH) 
    ]
    return sum(files, [])

## Grabbing each file and creating a merged dataframe

In [4]:
files = [f for f in grab_all_files(DATA_PATH, '.pt') if (not f.startswith('._')) and ('null-corpus' in f)]
print(files)

['../data/ckpts/graph-obj-null-corpus.parquet.pt']


In [5]:
df_ = []
for f in tqdm(files):
    ckpt = torch.load(f)
    df = pd.DataFrame(ckpt['meta_data'])
    
    # number of tokens per text
    df['nx'] = ckpt['N'][:,0].tolist()
    df['ny'] = ckpt['N'][:,1].tolist()
    
    # CE values
    df['Hxy'] = ckpt['M'][:,0].tolist()
    df['Hyx'] = ckpt['M'][:,1].tolist()
    
    df_ += [df]

df = pd.concat(df_, ignore_index=True)
print(df.shape)
df.head()

  0%|          | 0/1 [00:00<?, ?it/s]

(4110515, 34)


,document_line_no,x_turn_id,x_speaker,bullet,recipient,conversation_id,overlapping_text,corpus,com,exp,...,x_timestamp,x_speaker_loc,x_speaker_gender,y_timestamp,y_speaker_loc,y_speaker_gender,nx,ny,Hxy,Hyx
0,46.0,4175-19,callfriend-eng_n-4175-M1,"[28939, 36427]",ADULT,4175,0.0,callfriend-eng_n,None,None,...,None,None,None,None,None,None,29.0,25.0,8.396281,7.167986
1,46.0,4175-19,callfriend-eng_n-4175-M1,"[28939, 36427]",ADULT,4175,0.0,callfriend-eng_n,None,None,...,None,None,None,None,None,None,29.0,5.0,9.874767,1.079542
2,46.0,4175-19,callfriend-eng_n-4175-M1,"[28939, 36427]",ADULT,4175,0.0,callfriend-eng_n,None,None,...,None,None,None,None,None,None,29.0,16.0,8.867210,4.527740
3,46.0,4175-19,callfriend-eng_n-4175-M1,"[28939, 36427]",ADULT,4175,0.0,callfriend-eng_n,None,None,...,None,None,None,None,None,None,29.0,5.0,9.359125,1.006004
4,46.0,4175-19,callfriend-eng_n-4175-M1,"[28939, 36427]",ADULT,4175,0.0,callfriend-eng_n,None,None,...,None,None,None,None,None,None,29.0,3.0,9.678131,0.365403


In [6]:
list(df)

['document_line_no',
 'x_turn_id',
 'x_speaker',
 'bullet',
 'recipient',
 'conversation_id',
 'overlapping_text',
 'corpus',
 'com',
 'exp',
 'mor',
 'gra',
 'gpx',
 'act',
 'pic',
 'par',
 'sit',
 'raw_text',
 'y_speaker',
 'y_turn_id',
 'y_utterance_delta_no',
 'self',
 'sample_delta',
 'wor',
 'x_timestamp',
 'x_speaker_loc',
 'x_speaker_gender',
 'y_timestamp',
 'y_speaker_loc',
 'y_speaker_gender',
 'nx',
 'ny',
 'Hxy',
 'Hyx']

#### Creating Add'l Regression Model Variables

In [7]:
# df['age_dif'] = df['x_age'] - df['y_age']
# df['pol_dif'] = df['x_politics'] - df['y_politics']
# df['status_dif'] = df['x_i_think_my_status'] - df['x_i_think_your_status']
# df['same_gender'] = (df['x_sex'] == df['y_sex']).astype(int)
# df['same_race'] = (df['x_race'] == df['y_race']).astype(int)
# df['same_edu'] = (df['x_edu'] == df['y_edu']).astype(int)
# df['t_delta'] = df['y_turn_id'] - df['x_turn_id']

In [8]:
# df[['age_dif', 'pol_dif', 'status_dif', 'same_gender', 'same_race', 't_delta', 'same_edu']].mean()

In [9]:
# df[['age_dif', 'pol_dif', 'status_dif', 'same_gender', 'same_race', 't_delta', 'same_edu']].median()

In [10]:
# df[['age_dif', 'pol_dif', 'status_dif', 'same_gender', 'same_race', 't_delta', 'same_edu']].std()

## Save data

In [11]:
acceptable_cols = ['corpus', 'conversation_id', 'document_line_no', 'x_turn_id', 'x_speaker', 'y_speaker', 'y_turn_id', 'y_utterance_delta_no', 'self', 'nx', 'ny', 'Hxy', 'Hyx']

In [12]:
df = df[acceptable_cols]

In [13]:
for col in list(df):
    df[col] = [it.replace('\t', '') if isinstance(it,str) else it for it in tqdm(df[col])]

  0%|          | 0/4110515 [00:00<?, ?it/s]

  0%|          | 0/4110515 [00:00<?, ?it/s]

  0%|          | 0/4110515 [00:00<?, ?it/s]

  0%|          | 0/4110515 [00:00<?, ?it/s]

  0%|          | 0/4110515 [00:00<?, ?it/s]

...

  0%|          | 0/4110515 [00:00<?, ?it/s]

  0%|          | 0/4110515 [00:00<?, ?it/s]

  0%|          | 0/4110515 [00:00<?, ?it/s]

  0%|          | 0/4110515 [00:00<?, ?it/s]

  0%|          | 0/4110515 [00:00<?, ?it/s]

In [14]:
# df[acceptable_cols].to_csv(
#     OUTPUT_NAME,
#     index=False,
#     encoding='utf-8',
#     sep='\t'
# )

In [15]:
df[acceptable_cols].to_parquet(OUTPUT_NAME)